# 第五章圖表（Deployment / Test 版本）

本 notebook 只保留 **deployment / test** 階段的比較內容，不混入訓練時間與整批實驗流程時間。

資料來源分成兩部分：
- **傳統模型**：`results_model_matrix_0609.csv`
- **LLM**：`eb_article_inference_timing`


In [41]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'results_model_matrix_0609.csv').exists():
    BASE_DIR = Path(r'd:/NTPU_class/paper/code')

RESULT_CSV = BASE_DIR / 'results_model_matrix_0609.csv'
DETAIL_CSV = BASE_DIR / 'results_model_inference_detail_0609.csv'
DB_FILE = Path(r'd:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite')
PIC_DIR = Path(r'D:/NTPU_class/paper/pic')
PIC_DIR.mkdir(parents=True, exist_ok=True)

COLOR_MAP = {'jieba': '#4C78A8', 'ckip': '#E45756'}
STAGE_COLOR_MAP = {'Tokenize': '#72B7B2', 'Feature Build': '#F58518', 'Model Predict': '#54A24B'}
COMPARE_COLOR_MAP = {'最佳傳統模型': '#4C78A8', 'LLM': '#E45756'}


def prettify_repr(value):
    return {
        'tfidf': 'TF-IDF',
        'tfidf_svd300': 'TF-IDF+SVD',
        'w2v_mean': 'Word2Vec',
        'fasttext_mean': 'fastText',
    }.get(value, value)


def prettify_model(value):
    return {
        'lr': 'LR',
        'svm_rbf': 'SVM-RBF',
        'svm_poly': 'SVM-Poly',
        'rf': 'RF',
        'xgb': 'XGBoost',
        'mlp': 'MLP',
    }.get(value, value)


def apply_thesis_theme(fig, height=520, legend_title=None):
    fig.update_layout(
        template='plotly_white',
        height=height,
        font=dict(family='Microsoft JhengHei, Arial', size=14, color='#1F2937'),
        title=dict(font=dict(size=20, color='#111827'), x=0.5, xanchor='center'),
        margin=dict(l=80, r=40, t=90, b=70),
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1, title_text=legend_title),
    )
    fig.update_xaxes(showgrid=True, gridcolor='rgba(0,0,0,0.08)', zeroline=False, linecolor='rgba(0,0,0,0.15)')
    fig.update_yaxes(showgrid=True, gridcolor='rgba(0,0,0,0.08)', zeroline=False, linecolor='rgba(0,0,0,0.15)')
    return fig


def p95(series):
    s = pd.to_numeric(series, errors='coerce').dropna()
    return np.percentile(s, 95) if len(s) else np.nan

def export_fig(fig, filename, width=1200, height=700, scale=2):
    output_path = PIC_DIR / filename
    try:
        fig.write_image(str(output_path), width=width, height=height, scale=scale)
        print(f'exported: {output_path}')
    except Exception as err:
        print(f'圖片輸出失敗：{output_path}')
        print('請確認已安裝 kaleido，例如：pip install -U kaleido')
        print(err)


## A. 傳統模型（Deployment / Test）

本段只看已訓練模型在單篇文本輸入情境下的推論時間，不含訓練時間。


In [42]:
df_model = pd.read_csv(RESULT_CSV)

for col in [
    'acc', 'f1', 'roc_auc', 'pr_auc',
    'tokenize_mean_sec', 'feature_mean_sec', 'predict_mean_sec',
    'total_infer_mean_sec', 'total_infer_median_sec', 'total_infer_min_sec',
    'total_infer_max_sec', 'total_infer_p95_sec'
]:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors='coerce')

df_model['tokenizer'] = df_model['tokenizer'].astype(str).str.lower()
df_model['repr'] = df_model['repr'].astype(str).str.lower()
df_model['model'] = df_model['model'].astype(str).str.lower()
df_model['repr_pretty'] = df_model['repr'].map(prettify_repr)
df_model['model_pretty'] = df_model['model'].map(prettify_model)
df_model['combo'] = df_model['tokenizer'].str.upper() + ' | ' + df_model['repr_pretty'] + ' | ' + df_model['model_pretty']
df_model['tokenizer_repr'] = df_model['tokenizer'].str.upper() + ' | ' + df_model['repr_pretty']
df_model['efficiency'] = df_model['pr_auc'] / df_model['total_infer_mean_sec'].replace(0, np.nan)
df_model = df_model.sort_values(['pr_auc', 'roc_auc', 'f1'], ascending=False).reset_index(drop=True)

display(df_model.head(10))

if DETAIL_CSV.exists():
    df_detail = pd.read_csv(DETAIL_CSV)
    for col in ['text_length', 'tokenize_sec', 'feature_sec', 'predict_sec', 'total_infer_sec']:
        if col in df_detail.columns:
            df_detail[col] = pd.to_numeric(df_detail[col], errors='coerce')
    df_detail['tokenizer'] = df_detail['tokenizer'].astype(str).str.lower()
    df_detail['repr'] = df_detail['repr'].astype(str).str.lower()
    df_detail['model'] = df_detail['model'].astype(str).str.lower()
    df_detail['repr_pretty'] = df_detail['repr'].map(prettify_repr)
    df_detail['model_pretty'] = df_detail['model'].map(prettify_model)
    df_detail['combo'] = df_detail['tokenizer'].str.upper() + ' | ' + df_detail['repr_pretty'] + ' | ' + df_detail['model_pretty']
else:
    df_detail = pd.DataFrame()
    print('找不到 results_model_inference_detail_0609.csv，分布圖會略過傳統模型明細。')


,tokenizer,repr,model,prep_sec,feat_sec,train_sec,acc,f1,roc_auc,pr_auc,test_pred_sec,cov_train,cov_test,total_sec,n_texts,tokenize_mean_sec,feature_mean_sec,predict_mean_sec,total_infer_mean_sec,total_infer_median_sec,total_infer_min_sec,total_infer_max_sec,total_infer_p95_sec,text_length_mean,token_count_mean,repr_pretty,model_pretty,combo,tokenizer_repr,efficiency
0,ckip,fasttext_mean,svm_rbf,204.058941,3.432719,0.188101,0.700361,0.770083,0.903604,0.975405,0.029079,1.000000,1.000000,207.708839,277,0.154289,0.000842,0.001492,0.156623,0.054578,0.019096,1.384472,0.613301,216.592058,124.620939,fastText,SVM-RBF,CKIP | fastText | SVM-RBF,CKIP | fastText,6.227721
1,jieba,tfidf,lr,1.578999,0.445531,0.003299,0.801444,0.862843,0.901745,0.975289,0.000464,NaN,NaN,2.028293,277,0.002117,0.000857,0.000374,0.003349,0.001933,0.000782,0.021140,0.010912,216.592058,122.862816,TF-IDF,LR,JIEBA | TF-IDF | LR,JIEBA | TF-IDF,291.246837
2,jieba,w2v_mean,svm_rbf,1.578999,1.510796,0.204456,0.747292,0.813830,0.899685,0.975225,0.035495,0.934645,0.898995,3.329746,277,0.001897,0.000401,0.001166,0.003463,0.001867,0.000880,0.018726,0.011553,216.592058,122.862816,Word2Vec,SVM-RBF,JIEBA | Word2Vec | SVM-RBF,JIEBA | Word2Vec,281.604233
3,ckip,tfidf_svd300,xgb,204.058941,0.710974,2.057627,0.851986,0.908686,0.898109,0.975032,0.004341,NaN,NaN,206.831882,277,0.159542,0.002483,0.001858,0.163884,0.066112,0.025178,1.360378,0.623345,216.592058,124.620939,TF-IDF+SVD,XGBoost,CKIP | TF-IDF+SVD | XGBoost,CKIP | TF-IDF+SVD,5.949537
4,ckip,fasttext_mean,mlp,204.058941,3.432719,0.119805,0.855596,0.912664,0.900695,0.974641,0.001813,1.000000,1.000000,207.613277,277,0.153402,0.000859,0.001122,0.155383,0.051517,0.017933,1.390679,0.613935,216.592058,124.620939,fastText,MLP,CKIP | fastText | MLP,CKIP | fastText,6.272514
5,ckip,w2v_mean,svm_rbf,204.058941,1.279816,0.194388,0.736462,0.805333,0.899240,0.974485,0.030795,0.953589,0.919307,205.563939,277,0.154324,0.000484,0.001488,0.156296,0.052582,0.019289,1.371385,0.618318,216.592058,124.620939,Word2Vec,SVM-RBF,CKIP | Word2Vec | SVM-RBF,CKIP | Word2Vec,6.234868
6,ckip,w2v_mean,svm_poly,204.058941,1.279816,0.141978,0.722022,0.791328,0.898109,0.974295,0.011337,0.953589,0.919307,205.492071,277,0.154044,0.000485,0.001389,0.155919,0.052570,0.018200,1.371695,0.619928,216.592058,124.620939,Word2Vec,SVM-Poly,CKIP | Word2Vec | SVM-Poly,CKIP | Word2Vec,6.248724
7,ckip,fasttext_mean,lr,204.058941,3.432719,0.033527,0.765343,0.831169,0.900856,0.974289,0.001262,1.000000,1.000000,207.526448,277,0.160060,0.000849,0.001108,0.162017,0.058624,0.025085,1.386452,0.616158,216.592058,124.620939,fastText,LR,CKIP | fastText | LR,CKIP | fastText,6.013516
8,ckip,w2v_mean,lr,204.058941,1.279816,0.048457,0.776173,0.842640,0.899887,0.974257,0.001407,0.953589,0.919307,205.388620,277,0.159484,0.000497,0.001136,0.161118,0.060178,0.024911,1.366395,0.616113,216.592058,124.620939,Word2Vec,LR,CKIP | Word2Vec | LR,CKIP | Word2Vec,6.046841
9,ckip,tfidf,lr,204.058941,0.264325,0.003503,0.812274,0.870647,0.897786,0.973989,0.000389,NaN,NaN,204.327158,277,0.157671,0.001002,0.000423,0.159095,0.057731,0.024303,1.342482,0.606333,216.592058,124.620939,TF-IDF,LR,CKIP | TF-IDF | LR,CKIP | TF-IDF,6.122043


### 傳統模型效能與單篇推論時間摘要

**建議放置章節：5.3.1 整體分類表現比較**

說明：本表列出主要模型組合之分類效能與單篇部署推論平均時間，用以同時比較辨識能力與實際部署效率。


In [43]:
summary_model = df_model[[
    'combo', 'acc', 'f1', 'roc_auc', 'pr_auc', 'total_infer_mean_sec', 'total_infer_median_sec', 'total_infer_p95_sec'
]].head(10).copy()
summary_model.columns = ['模型組合', 'Accuracy', 'F1', 'ROC-AUC', 'PR-AUC', '單篇平均(秒)', '單篇中位數(秒)', '單篇P95(秒)']
summary_model = summary_model.round(6)
display(summary_model)

fig = go.Figure(data=[go.Table(
    header=dict(values=list(summary_model.columns), fill_color='#1F3A5F', font=dict(color='white', size=13), align='center', height=34),
    cells=dict(values=[summary_model[c] for c in summary_model.columns], fill_color=[['#F8FAFC', '#FFFFFF']], align='center', height=30, font=dict(size=12, color='#111827'))
)])
fig.update_layout(title='傳統模型效能與單篇推論時間摘要', height=420, margin=dict(l=20, r=20, t=70, b=20))
export_fig(fig, 'traditional_model_summary_table.png', width=1400, height=420, scale=2)
fig.show()


,模型組合,Accuracy,F1,ROC-AUC,PR-AUC,單篇平均(秒),單篇中位數(秒),單篇P95(秒)
0,CKIP | fastText | SVM-RBF,0.700361,0.770083,0.903604,0.975405,0.156623,0.054578,0.613301
1,JIEBA | TF-IDF | LR,0.801444,0.862843,0.901745,0.975289,0.003349,0.001933,0.010912
2,JIEBA | Word2Vec | SVM-RBF,0.747292,0.813830,0.899685,0.975225,0.003463,0.001867,0.011553
3,CKIP | TF-IDF+SVD | XGBoost,0.851986,0.908686,0.898109,0.975032,0.163884,0.066112,0.623345
4,CKIP | fastText | MLP,0.855596,0.912664,0.900695,0.974641,0.155383,0.051517,0.613935
5,CKIP | Word2Vec | SVM-RBF,0.736462,0.805333,0.899240,0.974485,0.156296,0.052582,0.618318
6,CKIP | Word2Vec | SVM-Poly,0.722022,0.791328,0.898109,0.974295,0.155919,0.052570,0.619928
7,CKIP | fastText | LR,0.765343,0.831169,0.900856,0.974289,0.162017,0.058624,0.616158
8,CKIP | Word2Vec | LR,0.776173,0.842640,0.899887,0.974257,0.161118,0.060178,0.616113
9,CKIP | TF-IDF | LR,0.812274,0.870647,0.897786,0.973989,0.159095,0.057731,0.606333


exported: D:\NTPU_class\paper\pic\traditional_model_summary_table.png


### 圖 5-1 PR-AUC 與單篇部署推論時間之權衡

**建議放置章節：5.3.3 最佳模型與應用情境討論**

圖說：本圖以單篇部署推論平均時間為橫軸、PR-AUC 為縱軸，比較各模型組合在分類效果與即時性之間的權衡關係。


In [66]:
fig = px.scatter(
    df_model,
    x='total_infer_mean_sec',
    y='pr_auc',
    color='tokenizer',
    symbol='model_pretty',
    hover_name='combo',
    hover_data={
        'acc': ':.3f',
        'f1': ':.3f',
        'roc_auc': ':.3f',
        'tokenize_mean_sec': ':.6f',
        'feature_mean_sec': ':.6f',
        'predict_mean_sec': ':.6f',
        'total_infer_mean_sec': ':.6f',
        'combo': False,
    },
    color_discrete_map=COLOR_MAP,
    title='PR-AUC 與單篇部署推論時間之權衡',
    labels={'total_infer_mean_sec': '單篇部署推論平均時間（秒）', 'pr_auc': 'PR-AUC', 'model_pretty': '模型'}
)
fig.update_traces(marker=dict(size=13, line=dict(width=1, color='white'), opacity=0.9))

fig.update_xaxes(type='log')

apply_thesis_theme(fig, height=680, legend_title='斷詞 / 模型')

fig.update_layout(
    height=680,
    margin=dict(l=90, r=60, t=70, b=100),
    title=dict(
        text='PR-AUC 與單篇部署推論時間之權衡',
        x=0.5,
        y=0.95,
        xanchor='center',
        yanchor='top'
    ),
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.12,
        xanchor='center',
        x=0.5,
        title_text='斷詞 / 模型'
    )
)
export_fig(fig, 'pr_auc_vs_inference_time.png', width=1200, height=620, scale=2)
fig.show()


exported: D:\NTPU_class\paper\pic\pr_auc_vs_inference_time.png


### 圖 5-2 高表現組合的單篇部署時間拆解（Top 6）

**建議放置章節：5.3.3 最佳模型與應用情境討論**

圖說：本圖針對高表現模型組合，拆解其單篇部署推論時間中的斷詞、特徵建構與模型預測三個階段，以說明時間成本結構。


In [45]:
top_time = df_model.sort_values(['pr_auc', 'roc_auc'], ascending=False).head(6).copy()
long_time = top_time.melt(
    id_vars=['combo'],
    value_vars=['tokenize_mean_sec', 'feature_mean_sec', 'predict_mean_sec'],
    var_name='stage',
    value_name='seconds'
)
long_time['stage'] = long_time['stage'].map({
    'tokenize_mean_sec': 'Tokenize',
    'feature_mean_sec': 'Feature Build',
    'predict_mean_sec': 'Model Predict',
})

fig = px.bar(
    long_time,
    x='seconds',
    y='combo',
    color='stage',
    color_discrete_map=STAGE_COLOR_MAP,
    orientation='h',
    title='高表現組合的單篇部署時間拆解（Top 6）',
    labels={'seconds': '單篇部署推論時間（秒）', 'combo': '模型組合', 'stage': '階段'}
)
fig.update_yaxes(automargin=True)
apply_thesis_theme(fig, height=620, legend_title='階段')
fig.update_layout(barmode='stack')
export_fig(fig, 'top6_time_breakdown.png', width=1200, height=620, scale=2)
fig.show()


exported: D:\NTPU_class\paper\pic\top6_time_breakdown.png


### 不同應用情境的建議模型組合

**建議放置章節：5.3.3 最佳模型與應用情境討論**

說明：本表整理離線分析與即時應用兩種情境下較適合的模型組合，並列出其主要效能指標與單篇部署推論平均時間。


In [46]:
offline_best = df_model.sort_values(['pr_auc', 'roc_auc', 'f1'], ascending=False).head(1).copy()
realtime_best = df_model[df_model['tokenizer'] == 'jieba'].sort_values(['pr_auc', 'total_infer_mean_sec'], ascending=[False, True]).head(1).copy()

scenario_table = pd.concat([offline_best, realtime_best], ignore_index=True)
scenario_table = scenario_table[['tokenizer', 'repr_pretty', 'model_pretty', 'acc', 'f1', 'roc_auc', 'pr_auc', 'total_infer_mean_sec']].copy()
scenario_table.insert(0, 'scenario', ['離線分析', '即時應用'])
scenario_table.columns = ['情境', '斷詞', '文本表示', '模型', 'Accuracy', 'F1', 'ROC-AUC', 'PR-AUC', '單篇平均(秒)']
scenario_table = scenario_table.round(6)
display(scenario_table)

fig = go.Figure(data=[go.Table(
    header=dict(values=list(scenario_table.columns), fill_color='#1F3A5F', font=dict(color='white', size=13), align='center', height=34),
    cells=dict(values=[scenario_table[c] for c in scenario_table.columns], fill_color=[['#F8FAFC', '#FFFFFF']], align='center', height=32, font=dict(size=12, color='#111827'))
)])
fig.update_layout(title='不同應用情境的建議模型組合', height=300, margin=dict(l=20, r=20, t=70, b=20))
export_fig(fig, 'scenario_model_recommendation_table.png', width=1300, height=300, scale=2)
fig.show()


,情境,斷詞,文本表示,模型,Accuracy,F1,ROC-AUC,PR-AUC,單篇平均(秒)
0,離線分析,ckip,fastText,SVM-RBF,0.700361,0.770083,0.903604,0.975405,0.156623
1,即時應用,jieba,TF-IDF,LR,0.801444,0.862843,0.901745,0.975289,0.003349


exported: D:\NTPU_class\paper\pic\scenario_model_recommendation_table.png


## B. 傳統模型與 LLM 單篇預測時間比較

本段只比較單篇預測時間：
- 傳統模型：`total_infer_mean_sec` 與 `results_model_inference_detail_0609.csv` 的逐篇明細
- LLM：`llm_response_time_sec`

除最佳模型與 LLM 的直接比較外，本段也加入前 10 名模型與 LLM 的比較，以提高圖表資訊量並支撐「傳統模型推論速度較快」的結論。


In [47]:
query_latest = """
WITH latest_run AS (
    SELECT run_tag
    FROM eb_article_inference_timing
    GROUP BY run_tag
    ORDER BY MAX(created_at) DESC
    LIMIT 1
)
SELECT
    t.run_tag,
    t.article_id,
    t.created_at,
    t.text_length,
    t.status,
    t.llm_response_time_sec,
    t.db_write_time_sec,
    a.title
FROM eb_article_inference_timing t
LEFT JOIN articles a ON a.id = t.article_id
WHERE t.run_tag = (SELECT run_tag FROM latest_run)
ORDER BY t.created_at ASC;
"""

try:
    conn = sqlite3.connect(DB_FILE)
    df_time = pd.read_sql_query(query_latest, conn)
    conn.close()
except Exception as err:
    df_time = pd.DataFrame()
    print('讀取 eb_article_inference_timing 失敗：', err)

if df_time.empty:
    print('尚未抓到 eb_article_inference_timing 資料，請先完成 rag.ipynb 重跑。')
else:
    for col in ['text_length', 'llm_response_time_sec', 'db_write_time_sec']:
        if col in df_time.columns:
            df_time[col] = pd.to_numeric(df_time[col], errors='coerce')
    display(df_time.head(10))

df_done = df_time[df_time['status'] == 'done'].copy() if not df_time.empty else pd.DataFrame()


,run_tag,article_id,created_at,text_length,status,llm_response_time_sec,db_write_time_sec,title
0,article_rerun_20260609_111948,68fce9b6af1137205015fd06,2026-06-09 11:19:56,397,done,4.883694,0.014553,被爸媽情緒勒索很痛苦 心理諮商有效嗎?
1,article_rerun_20260609_111948,68fce9f3af11377624f11eda,2026-06-09 11:20:05,2106,done,8.465717,0.011975,看完這篇其實 情緒勒索是不是也等於控制慾強?
2,article_rerun_20260609_111948,68fcea01af11377624f11edb,2026-06-09 11:20:10,143,done,4.153937,0.010922,沒有情緒勒索，只有忠言逆耳
3,article_rerun_20260609_111948,68fcea15af11377624f11edc,2026-06-09 11:20:17,1248,done,6.623693,0.010156,#書籍推薦 #不被情緒勒索的51個方法
4,article_rerun_20260609_111948,68fcea6eaf11379f9c929394,2026-06-09 11:20:21,63,done,3.533831,0.016701,有關情緒勒索
5,article_rerun_20260609_111948,68fcea81af11379f9c929395,2026-06-09 11:20:25,134,done,3.885234,0.012665,另一半的爸媽如果情緒勒索
6,article_rerun_20260609_111948,68fcea94af11379f9c929396,2026-06-09 11:20:29,78,done,3.531506,0.012973,台灣社會怎愈來愈多情緒勒索？
7,article_rerun_20260609_111948,68fceaa7af11379f9c929397,2026-06-09 11:20:34,48,done,4.054860,0.023664,颱風假 是情緒勒索或討好民眾？
8,article_rerun_20260609_111948,68fceabaaf11379f9c929398,2026-06-09 11:20:38,109,done,3.922572,0.015303,家人永無止盡的情緒勒索
9,article_rerun_20260609_111948,68fceae1af11379f9c929399,2026-06-09 11:20:43,12,done,3.632567,0.009961,以「為你好」之名情緒勒索、強迫兒女


### 圖 5-3 最佳傳統模型與 LLM 單篇預測時間比較圖

**建議放置章節：5.4.1 傳統模型與 LLM 推論時間比較**

圖說：本圖比較最佳傳統模型與 LLM 在單篇文本預測情境下的平均耗時，用以說明傳統模型在實務部署上是否具備較高即時性。


In [48]:
if df_done.empty or 'total_infer_mean_sec' not in df_model.columns:
    print('無可用比較資料。')
else:
    best_traditional = df_model.sort_values(['pr_auc', 'roc_auc', 'f1'], ascending=False).iloc[0].copy()
    llm_mean_sec = df_done['llm_response_time_sec'].mean()

    compare_df = pd.DataFrame({
        '方法類型': ['最佳傳統模型', 'LLM'],
        '顯示名稱': [f"最佳傳統模型\n{best_traditional['tokenizer'].upper()} | {best_traditional['repr_pretty']} | {best_traditional['model_pretty']}",
            'LLM'
        ],
        'seconds': [best_traditional['total_infer_mean_sec'], llm_mean_sec]
    })
    speedup = llm_mean_sec / best_traditional['total_infer_mean_sec']
    display(compare_df)

    fig = px.bar(
        compare_df,
        x='顯示名稱',
        y='seconds',
        color='方法類型',
        text='seconds',
        color_discrete_map=COMPARE_COLOR_MAP,
        title='最佳傳統模型與 LLM 單篇預測時間比較',
        labels={'顯示名稱': '方法', 'seconds': '單篇預測時間（秒）'}
    )
    fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')
    apply_thesis_theme(fig, height=500, legend_title='方法類型')
    fig.add_annotation(
        x=1,
        y=max(compare_df['seconds']) * 0.92,
        text=f"最佳傳統模型約比 LLM 快 {speedup:.1f} 倍",
        showarrow=False,
        font=dict(size=13, color='#111827'),
        bgcolor='rgba(255,255,255,0.85)'
    )
    export_fig(fig, 'best_traditional_vs_llm_time.png', width=1100, height=500, scale=2)
    fig.show()


,方法類型,顯示名稱,seconds
0,最佳傳統模型,最佳傳統模型\nCKIP | fastText | SVM-RBF,0.156623
1,LLM,LLM,6.544567


exported: D:\NTPU_class\paper\pic\best_traditional_vs_llm_time.png


### 圖 5-4 前 10 名傳統模型與 LLM 單篇預測時間比較圖

**建議放置章節：5.4.1 傳統模型與 LLM 推論時間比較**

圖說：本圖將 PR-AUC 前 10 名傳統模型與 LLM 的平均單篇預測時間放在同一張圖中比較，並在標籤中保留各模型 PR-AUC，使圖中同時呈現模型效果與推論速度資訊。


In [49]:
if df_done.empty:
    print('無可用 LLM 時間資料。')
else:
    top10_model = df_model.sort_values(['pr_auc', 'roc_auc', 'f1'], ascending=False).head(10).copy()
    llm_mean_sec = df_done['llm_response_time_sec'].mean()

    top10_compare = top10_model[['combo', 'pr_auc', 'roc_auc', 'f1', 'total_infer_mean_sec']].copy()
    top10_compare['method'] = top10_compare['combo']
    top10_compare['method_type'] = '傳統模型 Top 10'
    top10_compare['seconds'] = top10_compare['total_infer_mean_sec']
    top10_compare['label'] = top10_compare.apply(lambda r: f"{r['seconds']:.4f}s / PR-AUC {r['pr_auc']:.3f}", axis=1)

    llm_row = pd.DataFrame({
        'combo': ['LLM'],
        'pr_auc': [np.nan],
        'roc_auc': [np.nan],
        'f1': [np.nan],
        'total_infer_mean_sec': [llm_mean_sec],
        'method': ['LLM'],
        'method_type': ['LLM'],
        'seconds': [llm_mean_sec],
        'label': [f"{llm_mean_sec:.4f}s"]
    })

    top10_compare_plot = pd.concat([top10_compare, llm_row], ignore_index=True)
    top10_compare_plot = top10_compare_plot.sort_values('seconds', ascending=True)
    display(top10_compare_plot[['method', 'method_type', 'seconds', 'pr_auc', 'roc_auc', 'f1']].round(6))

    fig = px.bar(
        top10_compare_plot,
        x='seconds',
        y='method',
        color='method_type',
        text='label',
        orientation='h',
        color_discrete_map={'傳統模型 Top 10': '#4C78A8', 'LLM': '#E45756'},
        title='前 10 名傳統模型與 LLM 單篇預測時間比較',
        labels={'seconds': '單篇預測平均時間（秒）', 'method': '方法', 'method_type': '方法類型'}
    )
    fig.update_traces(textposition='outside', cliponaxis=False)
    fig.update_xaxes(type='log')
    fig.update_yaxes(automargin=True)
    apply_thesis_theme(fig, height=760, legend_title='方法類型')
    export_fig(fig, 'top10_traditional_vs_llm_time.png', width=1400, height=760, scale=2)
    fig.show()


,method,method_type,seconds,pr_auc,roc_auc,f1
1,JIEBA | TF-IDF | LR,傳統模型 Top 10,0.003349,0.975289,0.901745,0.862843
2,JIEBA | Word2Vec | SVM-RBF,傳統模型 Top 10,0.003463,0.975225,0.899685,0.813830
4,CKIP | fastText | MLP,傳統模型 Top 10,0.155383,0.974641,0.900695,0.912664
6,CKIP | Word2Vec | SVM-Poly,傳統模型 Top 10,0.155919,0.974295,0.898109,0.791328
5,CKIP | Word2Vec | SVM-RBF,傳統模型 Top 10,0.156296,0.974485,0.899240,0.805333
0,CKIP | fastText | SVM-RBF,傳統模型 Top 10,0.156623,0.975405,0.903604,0.770083
9,CKIP | TF-IDF | LR,傳統模型 Top 10,0.159095,0.973989,0.897786,0.870647
8,CKIP | Word2Vec | LR,傳統模型 Top 10,0.161118,0.974257,0.899887,0.842640
7,CKIP | fastText | LR,傳統模型 Top 10,0.162017,0.974289,0.900856,0.831169
3,CKIP | TF-IDF+SVD | XGBoost,傳統模型 Top 10,0.163884,0.975032,0.898109,0.908686


exported: D:\NTPU_class\paper\pic\top10_traditional_vs_llm_time.png


### 圖 5-5 前 10 名傳統模型相對 LLM 的速度倍率圖

**建議放置章節：5.4.1 傳統模型與 LLM 推論時間比較**

圖說：本圖以 LLM 平均單篇預測時間除以各傳統模型平均單篇預測時間，呈現前 10 名模型相對於 LLM 的速度倍率；倍率越高代表傳統模型相對 LLM 越快。


In [50]:
if df_done.empty:
    print('無可用 LLM 時間資料。')
else:
    top10_speed = df_model.sort_values(['pr_auc', 'roc_auc', 'f1'], ascending=False).head(10).copy()
    llm_mean_sec = df_done['llm_response_time_sec'].mean()
    top10_speed['speedup_vs_llm'] = llm_mean_sec / top10_speed['total_infer_mean_sec']
    top10_speed = top10_speed.sort_values('speedup_vs_llm', ascending=True)
    display(top10_speed[['combo', 'pr_auc', 'total_infer_mean_sec', 'speedup_vs_llm']].round(6))

    fig = px.bar(
        top10_speed,
        x='speedup_vs_llm',
        y='combo',
        color='tokenizer',
        color_discrete_map=COLOR_MAP,
        text='speedup_vs_llm',
        orientation='h',
        title='前 10 名傳統模型相對 LLM 的速度倍率',
        labels={'speedup_vs_llm': '相對 LLM 速度倍率（倍）', 'combo': '模型組合', 'tokenizer': '斷詞方法'}
    )
    fig.update_traces(texttemplate='%{text:.1f}x', textposition='outside', cliponaxis=False)
    fig.update_yaxes(automargin=True)
    apply_thesis_theme(fig, height=720, legend_title='斷詞方法')
    export_fig(fig, 'top10_speedup_vs_llm.png', width=1400, height=720, scale=2)
    fig.show()


,combo,pr_auc,total_infer_mean_sec,speedup_vs_llm
3,CKIP | TF-IDF+SVD | XGBoost,0.975032,0.163884,39.934235
7,CKIP | fastText | LR,0.974289,0.162017,40.394446
8,CKIP | Word2Vec | LR,0.974257,0.161118,40.619619
9,CKIP | TF-IDF | LR,0.973989,0.159095,41.136124
0,CKIP | fastText | SVM-RBF,0.975405,0.156623,41.785435
5,CKIP | Word2Vec | SVM-RBF,0.974485,0.156296,41.872910
6,CKIP | Word2Vec | SVM-Poly,0.974295,0.155919,41.974146
4,CKIP | fastText | MLP,0.974641,0.155383,42.118987
2,JIEBA | Word2Vec | SVM-RBF,0.975225,0.003463,1889.796638
1,JIEBA | TF-IDF | LR,0.975289,0.003349,1954.378270


exported: D:\NTPU_class\paper\pic\top10_speedup_vs_llm.png


### 圖 5-6 前 10 名傳統模型與 LLM 單篇預測時間分布箱型圖

**建議放置章節：5.4.2 單篇預測時間摘要比較**

圖說：本圖使用逐篇推論明細比較前 10 名傳統模型與 LLM 的單篇預測時間分布，可同時觀察中位數、離散程度與極端值，補足只看平均值可能忽略的穩定性資訊。


In [51]:
if df_done.empty or df_detail.empty:
    print('無可用逐篇時間明細，略過分布箱型圖。')
else:
    top10_combos = df_model.sort_values(['pr_auc', 'roc_auc', 'f1'], ascending=False).head(10)['combo'].tolist()
    traditional_dist = df_detail[df_detail['combo'].isin(top10_combos)].copy()
    traditional_dist = traditional_dist[['combo', 'total_infer_sec']].rename(columns={'combo': 'method', 'total_infer_sec': 'seconds'})
    traditional_dist['method_type'] = '傳統模型 Top 10'

    llm_dist = df_done[['llm_response_time_sec']].rename(columns={'llm_response_time_sec': 'seconds'}).copy()
    llm_dist['method'] = 'LLM'
    llm_dist['method_type'] = 'LLM'

    dist_plot = pd.concat([traditional_dist, llm_dist], ignore_index=True)
    display(dist_plot.groupby(['method_type', 'method'])['seconds'].agg(['count', 'mean', 'median', 'max']).reset_index().round(6))

    fig = px.box(
        dist_plot,
        x='seconds',
        y='method',
        color='method_type',
        points='outliers',
        color_discrete_map={'傳統模型 Top 10': '#4C78A8', 'LLM': '#E45756'},
        title='前 10 名傳統模型與 LLM 單篇預測時間分布',
        labels={'seconds': '單篇預測時間（秒）', 'method': '方法', 'method_type': '方法類型'}
    )
    fig.update_xaxes(type='log')
    fig.update_yaxes(automargin=True)
    apply_thesis_theme(fig, height=780, legend_title='方法類型')
    export_fig(fig, 'top10_traditional_llm_time_boxplot.png', width=1400, height=780, scale=2)
    fig.show()


,method_type,method,count,mean,median,max
0,LLM,LLM,1389,6.544567,6.117983,46.440193
1,傳統模型 Top 10,CKIP | TF-IDF | LR,277,0.159095,0.057731,1.342482
2,傳統模型 Top 10,CKIP | TF-IDF+SVD | XGBoost,277,0.163884,0.066112,1.360378
3,傳統模型 Top 10,CKIP | Word2Vec | LR,277,0.161118,0.060178,1.366395
4,傳統模型 Top 10,CKIP | Word2Vec | SVM-Poly,277,0.155919,0.052570,1.371695
5,傳統模型 Top 10,CKIP | Word2Vec | SVM-RBF,277,0.156296,0.052582,1.371385
6,傳統模型 Top 10,CKIP | fastText | LR,277,0.162017,0.058624,1.386452
7,傳統模型 Top 10,CKIP | fastText | MLP,277,0.155383,0.051517,1.390679
8,傳統模型 Top 10,CKIP | fastText | SVM-RBF,277,0.156623,0.054578,1.384472
9,傳統模型 Top 10,JIEBA | TF-IDF | LR,277,0.003349,0.001933,0.021140


exported: D:\NTPU_class\paper\pic\top10_traditional_llm_time_boxplot.png


### 傳統模型與 LLM 單篇預測時間摘要

**建議放置章節：5.4.2 單篇預測時間摘要比較**

說明：本表彙整最佳傳統模型與 LLM 的單篇預測時間統計量，以便直接比較兩者在平均值、中位數與 P95 指標上的差異。


In [52]:
if df_done.empty or 'total_infer_mean_sec' not in df_model.columns:
    print('無可用比較資料。')
else:
    best_traditional = df_model.sort_values(['pr_auc', 'roc_auc', 'f1'], ascending=False).iloc[0].copy()

    compare_summary = pd.DataFrame({
        '方法': ['最佳傳統模型', 'LLM'],
        '組合 / 來源': [f" {best_traditional['tokenizer'].upper()} | {best_traditional['repr_pretty']} | {best_traditional['model_pretty']}",'eb_article_inference_timing'],
        '平均值(秒)': [best_traditional['total_infer_mean_sec'], df_done['llm_response_time_sec'].mean()],
        '中位數(秒)': [best_traditional['total_infer_median_sec'], df_done['llm_response_time_sec'].median()],
        '最小值(秒)': [best_traditional['total_infer_min_sec'], df_done['llm_response_time_sec'].min()],
        '最大值(秒)': [best_traditional['total_infer_max_sec'], df_done['llm_response_time_sec'].max()],
        'P95(秒)': [best_traditional['total_infer_p95_sec'], p95(df_done['llm_response_time_sec'])],
    }).round(6)
    display(compare_summary)

    fig = go.Figure(data=[go.Table(
        header=dict(values=list(compare_summary.columns), fill_color='#1F3A5F', font=dict(color='white', size=13), align='center', height=34),
        cells=dict(values=[compare_summary[c] for c in compare_summary.columns], fill_color=[['#F8FAFC', '#FFFFFF']], align='center', height=32, font=dict(size=12, color='#111827'))
    )])
    fig.update_layout(title='傳統模型與 LLM 單篇預測時間摘要', height=300, margin=dict(l=20, r=20, t=70, b=20))
    export_fig(fig, 'traditional_llm_time_summary_table.png', width=1300, height=300, scale=2)
    fig.show()


,方法,組合 / 來源,平均值(秒),中位數(秒),最小值(秒),最大值(秒),P95(秒)
0,最佳傳統模型,CKIP | fastText | SVM-RBF,0.156623,0.054578,0.019096,1.384472,0.613301
1,LLM,eb_article_inference_timing,6.544567,6.117983,3.428391,46.440193,10.094535


exported: D:\NTPU_class\paper\pic\traditional_llm_time_summary_table.png


### LLM 單篇預測資料摘要

**建議放置章節：5.4.2 單篇預測時間摘要比較**

說明：本表整理本次 LLM 推論批次的有效樣本數與資料完整性，作為時間分析樣本基礎說明。


In [53]:
if df_time.empty:
    print('無可用時間資料。')
else:
    llm_run_summary = pd.DataFrame({
        '項目': ['run_tag', '總筆數', '成功筆數', '失敗筆數'],
        '值': [
            df_time['run_tag'].iloc[0] if 'run_tag' in df_time.columns and len(df_time) else '',
            len(df_time),
            int((df_time['status'] == 'done').sum()),
            int((df_time['status'] == 'error').sum()),
        ]
    })
    display(llm_run_summary)

    fig = go.Figure(data=[go.Table(
        header=dict(values=list(llm_run_summary.columns), fill_color='#1F3A5F', font=dict(color='white', size=13), align='center', height=34),
        cells=dict(values=[llm_run_summary[c] for c in llm_run_summary.columns], fill_color=[['#F8FAFC', '#FFFFFF']], align='center', height=32, font=dict(size=12, color='#111827'))
    )])
    fig.update_layout(title='LLM 單篇預測資料摘要', height=260, margin=dict(l=20, r=20, t=70, b=20))
    export_fig(fig, 'llm_prediction_run_summary_table.png', width=900, height=260, scale=2)
    fig.show()


,項目,值
0,run_tag,article_rerun_20260609_111948
1,總筆數,1389
2,成功筆數,1389
3,失敗筆數,0


exported: D:\NTPU_class\paper\pic\llm_prediction_run_summary_table.png


## 備註

- 本 notebook 已移除訓練時間與整批流程時間作為核心比較依據。
- 傳統模型只看 `total_infer_mean_sec` 與其拆解欄位。
- LLM 只看 `llm_response_time_sec` 作為單篇預測時間。
- 若學校格式將表格與圖分開編號，則目前的表可直接改列為表號。
